# DigitExtractor — YOLOv8 Training on Google Colab

Train the YOLO detector produced by **DigitExtractor** entirely in the cloud.

### What you need on Google Drive
Upload your DigitExtractor output folder. It must contain:

| Folder / File | Contents |
|---|---|
| `ROI_640/` | 640 × 640 PNG images |
| `ROI_640_labels/` | Matching YOLO `.txt` label files |
| `yolo_classes.txt` *(auto-detected)* | One class name per line |
| `yolo_class_map.json` *(auto-detected)* | Exported by DigitExtractor |

### Before you start
1. **Runtime → Change runtime type → GPU (T4)**  
2. Edit the **Configuration** cell below.  
3. Run all cells top-to-bottom (`Runtime → Run all`).


## 1 · GPU Check

In [1]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No GPU detected.\n'
        'Go to:  Runtime → Change runtime type → Hardware accelerator → GPU (T4)'
    )
print('✅ GPU available')
print(result.stdout.split('\n')[8])   # one-liner GPU summary

assert torch.cuda.is_available(), 'PyTorch cannot see CUDA — try restarting the runtime.'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


✅ GPU available
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
GPU : Tesla T4
VRAM: 15.6 GB


## 2 · Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3 · Install Dependencies

In [3]:
%pip install ultralytics -q
import ultralytics
ultralytics.checks()


Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


## 4 · Configuration
> **Edit the values in this cell before running anything else.**


In [4]:
import os

# ── Path to your DigitExtractor output folder on Google Drive ─────────────────
# Must contain ROI_640/ and ROI_640_labels/ sub-folders.
DATASET_PATH = '/content/drive/MyDrive/DigitExtractor_Output/TrainingFiles'   # ← CHANGE THIS

# ── Model ─────────────────────────────────────────────────────────────────────
# YOLOv8 size: 'n' nano | 's' small | 'm' medium | 'l' large | 'x' xlarge
# Recommendation:
#   Colab free  (T4  16 GB) → 's'   ~1–2 h for 100 epochs
#   Colab Pro   (A100 40 GB)→ 'm'   ~1 h  for 100 epochs
MODEL_SIZE = 's'

# ── Training hyper-parameters ─────────────────────────────────────────────────
EPOCHS     = 100          # total training epochs
BATCH_SIZE = 16           # 16 for T4, 32 for A100
VAL_SPLIT  = 0.15         # fraction of data held out for validation
SEED       = 42           # random seed for reproducible split

# ── Where to save the final results on Google Drive ───────────────────────────
RESULTS_DRIVE_PATH = '/content/drive/MyDrive/DigitExtractor_Output/Outputs'

# ─────────────────────────────────────────────────────────────────────────────
print('Configuration')
print(f'  Dataset   : {DATASET_PATH}')
print(f'  Model     : YOLOv8{MODEL_SIZE}')
print(f'  Epochs    : {EPOCHS}')
print(f'  Batch     : {BATCH_SIZE}')
print(f'  Val split : {int(VAL_SPLIT*100)} %')
print(f'  Results   : {RESULTS_DRIVE_PATH}')


Configuration
  Dataset   : /content/drive/MyDrive/DigitExtractor_Output
  Model     : YOLOv8s
  Epochs    : 100
  Batch     : 16
  Val split : 15 %
  Results   : /content/drive/MyDrive/DigitExtractor_Output/Training_Results


## 5 · Validate Dataset

In [5]:
import glob

IMAGES_DIR = os.path.join(DATASET_PATH, 'ROI_640')
LABELS_DIR = os.path.join(DATASET_PATH, 'ROI_640_labels')

if not os.path.isdir(IMAGES_DIR):
    raise FileNotFoundError(
        f'ROI_640/ not found at:\n  {IMAGES_DIR}\n'
        'Make sure DATASET_PATH points to your DigitExtractor output folder.'
    )
if not os.path.isdir(LABELS_DIR):
    raise FileNotFoundError(
        f'ROI_640_labels/ not found at:\n  {LABELS_DIR}\n'
        'Make sure DATASET_PATH points to your DigitExtractor output folder.'
    )

# collect all images and labels
all_imgs = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.bmp'):
    all_imgs.extend(glob.glob(os.path.join(IMAGES_DIR, ext)))
all_lbls = glob.glob(os.path.join(LABELS_DIR, '*.txt'))

img_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in all_imgs}
lbl_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in all_lbls}

paired          = sorted(set(img_by_stem) & set(lbl_by_stem))
orphan_imgs     = set(img_by_stem) - set(lbl_by_stem)
orphan_lbls     = set(lbl_by_stem) - set(img_by_stem)

print(f'Images found       : {len(all_imgs)}')
print(f'Label files found  : {len(all_lbls)}')
print(f'Matched pairs      : {len(paired)}')
if orphan_imgs:
    print(f'⚠  Images without labels : {len(orphan_imgs)}  (will be skipped)')
if orphan_lbls:
    print(f'⚠  Labels without images : {len(orphan_lbls)}  (will be skipped)')

if not paired:
    raise ValueError(
        'No matched image-label pairs found.\n'
        'Check that your filenames match between ROI_640/ and ROI_640_labels/.'
    )

print(f'\n✅  {len(paired)} samples ready.')


Images found       : 1292
Label files found  : 1308
Matched pairs      : 1292
⚠  Labels without images : 16  (will be skipped)

✅  1292 samples ready.


## 6 · Grayscale Preprocessing

Convert every matched image to **3-channel grayscale** (`BGR → GRAY → BGR`) before the train/val split.
Keeping 3 channels ensures full compatibility with YOLOv8's default loader while training exclusively on luminance information — matching what the tester does at inference time.

The `img_by_stem` mapping is updated in-place so all subsequent cells automatically use the grayscale copies without any further changes.

In [ ]:
import cv2 as _cv2
import numpy as _np

GRAY_DIR = '/content/drive/MyDrive/DigitExtractor_Output/Grayscales'
os.makedirs(GRAY_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Robust image reader — three methods so nothing is silently skipped.
# cv2.imread() returns None for paths with non-ASCII or special characters,
# which is exactly what was causing most Drive images to be skipped before.
# ---------------------------------------------------------------------------
def _read_img(path):
    # 1) np.fromfile + imdecode: reads raw bytes first, bypasses path issues
    try:
        raw = _np.fromfile(path, dtype=_np.uint8)
        if raw.size > 0:
            img = _cv2.imdecode(raw, _cv2.IMREAD_COLOR)
            if img is not None:
                return img
    except Exception:
        pass
    # 2) PIL / Pillow: broadest format support, handles anything GIMP can open
    try:
        from PIL import Image
        pil = Image.open(path).convert('RGB')
        arr = _np.array(pil)
        return _cv2.cvtColor(arr, _cv2.COLOR_RGB2BGR)
    except Exception:
        pass
    # 3) plain cv2.imread as absolute last resort
    return _cv2.imread(path, _cv2.IMREAD_COLOR)

total   = len(img_by_stem)
_done   = 0
_failed = []

print(f'Converting {total} images to 3-channel grayscale...')
print(f'Output  ->  {GRAY_DIR}')

for stem, src_path in list(img_by_stem.items()):
    # Always save as .png regardless of source format
    dst_name = os.path.splitext(os.path.basename(src_path))[0] + '.png'
    dst_path = os.path.join(GRAY_DIR, dst_name)

    img = _read_img(src_path)
    if img is None:
        _failed.append(src_path)
        continue

    gray     = _cv2.cvtColor(img, _cv2.COLOR_BGR2GRAY)
    gray_3ch = _cv2.cvtColor(gray, _cv2.COLOR_GRAY2BGR)

    # imencode + tofile: also path-safe for writing
    ok, buf = _cv2.imencode('.png', gray_3ch)
    if not ok:
        _failed.append(f'{src_path}  (encode failed)')
        continue

    buf.tofile(dst_path)
    img_by_stem[stem] = dst_path   # redirect: split cell will copy this path
    _done += 1

IMAGES_DIR = GRAY_DIR  # keep in sync with later references

print(f'Converted : {_done} / {total}')
if _failed:
    print(f'Failed    : {len(_failed)}')
    for p in _failed[:15]:
        print(f'  {p}')
    if len(_failed) > 15:
        print(f'  ... and {len(_failed) - 15} more')

# ── Hard check: every img_by_stem entry must now point to GRAY_DIR ───────────
_still_color = [p for p in img_by_stem.values() if GRAY_DIR not in p]
if _still_color:
    raise RuntimeError(
        f'{len(_still_color)} image(s) were NOT converted and still point to '
        f'the original ROI_640 directory.\n'
        f'Fix the failures above before continuing — otherwise training will '
        f'mix color and grayscale images.'
    )

print(f'All {len(img_by_stem)} entries in img_by_stem -> {GRAY_DIR}')
print('Training WILL use grayscale. Safe to run Split cell.')


## 7 · Detect Class List

In [ ]:
import json

# Built-in defaults — match DigitExtractor's class scheme exactly.
# Class 0  : digit_strip      (the full 5-digit strip bounding box)
# Class 1-10: digit_0 … digit_9 (individual digit boxes)
# Class 11 : digit_unreadable  (slots labelled X)
_DEFAULT_NAMES = [
    'digit_strip',
    'digit_0', 'digit_1', 'digit_2', 'digit_3', 'digit_4',
    'digit_5', 'digit_6', 'digit_7', 'digit_8', 'digit_9',
    'digit_unreadable',
]

class_names = None

# 1) yolo_classes.txt (written by DigitExtractor on every save)
_classes_txt = os.path.join(DATASET_PATH, 'yolo_classes.txt')
if os.path.isfile(_classes_txt):
    with open(_classes_txt) as f:
        lines = [l.strip() for l in f if l.strip()]
    if lines:
        class_names = lines
        print(f'✅  Loaded {len(class_names)} classes from yolo_classes.txt')

# 2) yolo_class_map.json
if class_names is None:
    _map_json = os.path.join(DATASET_PATH, 'yolo_class_map.json')
    if os.path.isfile(_map_json):
        with open(_map_json) as f:
            cmap = json.load(f)
        class_names = [cmap[str(i)]['name'] for i in range(len(cmap))]
        print(f'✅  Loaded {len(class_names)} classes from yolo_class_map.json')

# 3) fall back to built-in defaults
if class_names is None:
    class_names = _DEFAULT_NAMES.copy()
    print(f'ℹ️   No class file found — using built-in defaults ({len(class_names)} classes)')

# Scan labels to find the actual highest class ID in use
max_id_seen = 0
for stem in paired:
    with open(lbl_by_stem[stem]) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                max_id_seen = max(max_id_seen, int(parts[0]))

needed = max_id_seen + 1
if needed > len(class_names):
    print(f'⚠  Labels reference class {max_id_seen} but class list has only {len(class_names)} entries.')
    print('   Extending with placeholder names...')
    while len(class_names) < needed:
        class_names.append(f'class_{len(class_names)}')

NC = len(class_names)
print(f'\nFinal class list  ({NC} total):')
for i, name in enumerate(class_names):
    print(f'  {i:2d}  {name}')


## 8 · Train / Val Split

In [ ]:
import random, shutil

random.seed(SEED)
stems = paired[:]
random.shuffle(stems)

n_val   = max(1, round(len(stems) * VAL_SPLIT))
n_train = len(stems) - n_val
val_set   = set(stems[:n_val])
train_set = set(stems[n_val:])

print(f'Train : {n_train}')
print(f'Val   : {n_val}')

# Build the directory tree expected by ultralytics
WORK_DIR = '/content/yolo_dataset'
for split in ('train', 'val'):
    os.makedirs(os.path.join(WORK_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR, 'labels', split), exist_ok=True)

def _copy(stem, split):
    shutil.copy2(img_by_stem[stem],
                 os.path.join(WORK_DIR, 'images', split,
                              os.path.basename(img_by_stem[stem])))
    shutil.copy2(lbl_by_stem[stem],
                 os.path.join(WORK_DIR, 'labels', split,
                              os.path.basename(lbl_by_stem[stem])))

for s in train_set: _copy(s, 'train')
for s in val_set:   _copy(s, 'val')

print(f'\n✅  Files copied to {WORK_DIR}')


## 9 · Create data.yaml

In [ ]:
import yaml

DATA_YAML = os.path.join(WORK_DIR, 'data.yaml')

cfg = {
    'path' : WORK_DIR,
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : NC,
    'names': class_names,
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print('✅  data.yaml written:')
print(open(DATA_YAML).read())


## 10 · Train

> **Expected time on Colab T4 (free tier):**  
> ~50–90 min for 100 epochs on a typical DigitExtractor dataset.  
> Early-stopping (`patience=20`) will stop automatically if the model plateaus.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO(f'yolov8{MODEL_SIZE}.pt')

model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = 640,
    batch       = BATCH_SIZE,
    project     = '/content/runs',
    name        = 'digit_extractor',
    optimizer   = 'AdamW',
    lr0         = 0.001,
    lrf         = 0.01,
    weight_decay= 0.0005,
    warmup_epochs= 3,
    patience    = 20,          # early-stop if no improvement for 20 epochs
    save        = True,
    save_period = 10,          # checkpoint every 10 epochs
    plots       = True,
    verbose     = True,
    device      = 0,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'

print(f'\n✅  Training complete')
print(f'   Run dir  : {RUN_DIR}')
print(f'   Best .pt : {BEST_PT}')
print(f'   Last .pt : {LAST_PT}')


## 11 · Evaluate Best Model

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics    = best_model.val(data=DATA_YAML, imgsz=640, device=0)

print('\n── Validation Metrics ──────────────────────────────')
print(f'mAP@50      : {metrics.box.map50:.4f}')
print(f'mAP@50-95   : {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f}')
print(f'Recall      : {metrics.box.mr:.4f}')

# Per-class breakdown
print('\n── Per-class mAP@50 ────────────────────────────────')
if hasattr(metrics.box, 'ap_class_index') and metrics.box.ap_class_index is not None:
    for idx, cls_idx in enumerate(metrics.box.ap_class_index):
        name = class_names[cls_idx] if cls_idx < len(class_names) else f'class_{cls_idx}'
        ap   = metrics.box.ap50[idx] if idx < len(metrics.box.ap50) else float('nan')
        print(f'  {cls_idx:2d}  {name:<22s}  {ap:.4f}')


## 12 · Export to ONNX *(optional)*

Uncomment this cell if you want a portable ONNX model for deployment or inference
outside of PyTorch.


In [ ]:
# ── Uncomment to export ───────────────────────────────────────────────────────
# best_model.export(format='onnx', imgsz=640, dynamic=True, simplify=True)
# onnx_path = str(BEST_PT).replace('.pt', '.onnx')
# print(f'✅  Exported: {onnx_path}')


## 13 · Save Results to Google Drive

In [ ]:
import shutil

os.makedirs(RESULTS_DRIVE_PATH, exist_ok=True)
dest = os.path.join(RESULTS_DRIVE_PATH, RUN_DIR.name)
shutil.copytree(str(RUN_DIR), dest, dirs_exist_ok=True)

print('✅  Results saved to Google Drive:')
print(f'   {dest}')
print()
print('Key files:')
print(f'  best weights  →  {os.path.join(dest, "weights", "best.pt")}')
print(f'  last weights  →  {os.path.join(dest, "weights", "last.pt")}')
print(f'  metrics CSV   →  {os.path.join(dest, "results.csv")}')
print(f'  training plot →  {os.path.join(dest, "results.png")}')
